# Chess RL vs Stockfish - Lc0 Inspired
Deep ResNet architecture, proper loss functions, curriculum learning

In [ ]:
!apt-get update && apt-get install -y stockfish -qq
!pip install python-chess torch numpy matplotlib -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import chess
import chess.engine
import chess.pgn
import numpy as np
import json
from pathlib import Path
import random
import matplotlib.pyplot as plt
from datetime import datetime

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = Path('/content/drive/MyDrive/chess_lc0')
drive_path.mkdir(parents=True, exist_ok=True)
print(f'Path: {drive_path}')

## Board Encoder (13-plane simple)

In [ ]:
class BoardEncoder:
    def encode(self, board):
        tensor = torch.zeros(13, 8, 8, dtype=torch.float32)
        for sq in range(64):
            p = board.piece_at(sq)
            if p:
                r, c = sq // 8, sq % 8
                idx = p.piece_type - 1
                plane = idx if p.color == chess.WHITE else idx + 6
                tensor[plane, r, c] = 1.0
        if board.turn == chess.BLACK:
            tensor[12, :, :] = 1.0
        return tensor

encoder = BoardEncoder()
print(f'Encoder ready')

## Lc0-Inspired Deep Network

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, filters):
        super().__init__()
        self.conv1 = nn.Conv2d(filters, filters, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(filters)
        self.conv2 = nn.Conv2d(filters, filters, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(filters)
    
    def forward(self, x):
        residual = x
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return torch.relu(out + residual)

class ChessNetLc0(nn.Module):
    def __init__(self, filters=128, num_blocks=10):
        super().__init__()
        self.input_conv = nn.Conv2d(13, filters, kernel_size=3, padding=1)
        self.input_bn = nn.BatchNorm2d(filters)
        self.res_blocks = nn.Sequential(*[ResidualBlock(filters) for _ in range(num_blocks)])
        
        self.value_conv = nn.Conv2d(filters, 32, kernel_size=1)
        self.value_bn = nn.BatchNorm2d(32)
        self.value_fc1 = nn.Linear(32 * 64, 256)
        self.value_fc2 = nn.Linear(256, 1)
        
        self.policy_conv = nn.Conv2d(filters, 64, kernel_size=3, padding=1)
        self.policy_bn = nn.BatchNorm2d(64)
        self.policy_fc = nn.Linear(64 * 64, 4672)
    
    def forward(self, x):
        x = torch.relu(self.input_bn(self.input_conv(x)))
        x = self.res_blocks(x)
        
        v = torch.relu(self.value_bn(self.value_conv(x)))
        v = v.view(v.size(0), -1)
        v = torch.relu(self.value_fc1(v))
        value = torch.tanh(self.value_fc2(v))
        
        p = torch.relu(self.policy_bn(self.policy_conv(x)))
        p = p.view(p.size(0), -1)
        policy = torch.softmax(self.policy_fc(p), dim=1)
        
        return value, policy

net = ChessNetLc0(filters=128, num_blocks=10).to(device)
params = sum(p.numel() for p in net.parameters())
print(f'Network: {params:,} parameters (deep enough to overfit)')

## Greedy Move Selection

In [ ]:
def select_best_move(board, network, encoder):
    legal_moves = list(board.legal_moves)
    if not legal_moves:
        return None, 0.0
    
    best_move = None
    best_value = -float('inf')
    
    for move in legal_moves:
        board.push(move)
        tensor = encoder.encode(board).unsqueeze(0).to(device)
        
        with torch.no_grad():
            value, _ = network(tensor)
            val_score = value.item()
        
        board.pop()
        
        if val_score > best_value:
            best_value = val_score
            best_move = move
    
    return best_move, best_value

print('Move selector ready')

## Stockfish Engine

In [ ]:
class StockfishBot:
    def __init__(self, depth=1):
        self.engine = chess.engine.SimpleEngine.popen_uci('/usr/games/stockfish')
        self.depth = depth
    
    def get_move(self, board):
        try:
            result = self.engine.play(board, chess.engine.Limit(depth=self.depth))
            return result.move
        except Exception as e:
            if list(board.legal_moves):
                return random.choice(list(board.legal_moves))
            return None
    
    def set_depth(self, depth):
        self.depth = depth
    
    def quit(self):
        try:
            self.engine.quit()
        except:
            pass

print('Stockfish ready')

## Game Loop with Logging

In [ ]:
def play_game(network, encoder, stockfish, nn_is_white=True, max_moves=200, verbose=False):
    board = chess.Board()
    moves_data = []
    move_count = 0
    
    for _ in range(max_moves):
        if board.is_game_over():
            result = board.result()
            if verbose:
                print(f'  Result: {result} (NN={nn_is_white})')
            
            if result == '1-0':
                winner = 0 if nn_is_white else 1
            elif result == '0-1':
                winner = 1 if nn_is_white else 0
            else:
                winner = 2
            
            if verbose:
                print(f'  Winner code: {winner} (0=NN white, 1=NN black, 2=draw)')
            
            return winner, moves_data, move_count
        
        is_nn_turn = (board.turn == chess.WHITE) if nn_is_white else (board.turn == chess.BLACK)
        
        if is_nn_turn:
            move, value = select_best_move(board, network, encoder)
            if move is None:
                if verbose:
                    print(f'  NN has no legal moves!')
                return 1, moves_data, move_count
            
            board_state = encoder.encode(board).clone()
            board.push(move)
            moves_data.append({
                'board': board_state,
                'move': move,
                'nn_value': value
            })
            
            if verbose:
                print(f'  Move {move_count+1}: NN plays {move} (value={value:.4f})')
        else:
            move = stockfish.get_move(board)
            if move is None:
                if verbose:
                    print(f'  SF has no legal moves!')
                return 0, moves_data, move_count
            
            board.push(move)
            
            if verbose:
                print(f'  Move {move_count+1}: SF plays {move}')
        
        move_count += 1
    
    if verbose:
        print(f'  Game ended by max moves (draw)')
    return 2, moves_data, move_count

print('Game loop ready')

## Training with Correct Loss Functions

In [ ]:
def train_on_game(network, optimizer, moves_data, winner, learning_rate=0.001):
    if not moves_data:
        return 0.0, 0.0, 0.0
    
    network.train()
    total_value_loss = 0.0
    total_policy_loss = 0.0
    batch_count = 0
    batch_size = 4
    
    for i in range(0, len(moves_data), batch_size):
        batch_moves = moves_data[i:i+batch_size]
        boards_tensor = torch.stack([m['board'] for m in batch_moves]).to(device)
        
        values, policies = network(boards_tensor)
        
        if winner == 0 or winner == 1:
            target_value = 1.0
            value_weight = 1.0
        elif winner == 2:
            target_value = 0.0
            value_weight = 0.5
        else:
            target_value = -1.0
            value_weight = 1.0
        
        target_values = torch.tensor([target_value] * len(batch_moves), dtype=torch.float32).unsqueeze(1).to(device)
        
        value_loss = nn.MSELoss()(values, target_values) * value_weight
        
        policy_uniform = torch.ones_like(policies) / policies.size(1)
        policy_loss = nn.KLDivLoss(reduction='batchmean')(torch.log(policies + 1e-8), policy_uniform)
        
        total_loss = value_loss + policy_loss * 0.01
        
        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(network.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_value_loss += value_loss.item()
        total_policy_loss += policy_loss.item()
        batch_count += 1
    
    network.eval()
    
    avg_value_loss = total_value_loss / batch_count if batch_count > 0 else 0.0
    avg_policy_loss = total_policy_loss / batch_count if batch_count > 0 else 0.0
    avg_total_loss = avg_value_loss + avg_policy_loss * 0.01
    
    return avg_total_loss, avg_value_loss, avg_policy_loss

print('Training ready')

## Save/Load Checkpoints

In [ ]:
def save_checkpoint(network, optimizer, episode, depth, drive_path):
    checkpoint = {
        'episode': episode,
        'depth': depth,
        'model_state': network.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'timestamp': datetime.now().isoformat()
    }
    path = drive_path / f'ckpt_ep{episode}_d{depth}.pt'
    torch.save(checkpoint, path)
    return path

def load_checkpoint(path):
    checkpoint = torch.load(path, map_location=device)
    network = ChessNetLc0(filters=128, num_blocks=10).to(device)
    optimizer = optim.Adam(network.parameters(), lr=0.001)
    network.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    return network, optimizer, checkpoint['episode'], checkpoint['depth']

print('Checkpoints ready')

## Statistics Tracker

In [ ]:
class Stats:
    def __init__(self, drive_path):
        self.drive_path = drive_path
        self.file = drive_path / 'stats.json'
        self.stats = {
            'total_games': 0,
            'depths': {},
            'total_w': 0,
            'total_l': 0,
            'total_d': 0,
            'history': [],
            'loss_history': []
        }
        if self.file.exists():
            with open(self.file) as f:
                self.stats = json.load(f)
    
    def record(self, winner, depth, total_loss, value_loss, policy_loss, game_length, nn_value):
        self.stats['total_games'] += 1
        self.stats['loss_history'].append({
            'total': total_loss,
            'value': value_loss,
            'policy': policy_loss,
            'nn_value': nn_value
        })
        
        depth_str = str(depth)
        if depth_str not in self.stats['depths']:
            self.stats['depths'][depth_str] = {'w': 0, 'l': 0, 'd': 0, 'games': 0, 'lengths': []}
        
        if winner == 0 or winner == 1:
            self.stats['depths'][depth_str]['w'] += 1
            self.stats['total_w'] += 1
            self.stats['history'].append('W')
        elif winner == 2:
            self.stats['depths'][depth_str]['d'] += 1
            self.stats['total_d'] += 1
            self.stats['history'].append('D')
        else:
            self.stats['depths'][depth_str]['l'] += 1
            self.stats['total_l'] += 1
            self.stats['history'].append('L')
        
        self.stats['depths'][depth_str]['games'] += 1
        self.stats['depths'][depth_str]['lengths'].append(game_length)
        self.stats['history'] = self.stats['history'][-100:]
    
    def get_wr(self, depth):
        d_str = str(depth)
        if d_str not in self.stats['depths']:
            return 0.0
        d = self.stats['depths'][d_str]
        t = d['w'] + d['l'] + d['d']
        return d['w'] / t if t > 0 else 0.0
    
    def save(self):
        with open(self.file, 'w') as f:
            json.dump(self.stats, f, indent=2)
    
    def print_depth(self, depth):
        d_str = str(depth)
        if d_str not in self.stats['depths']:
            return
        d = self.stats['depths'][d_str]
        t = d['w'] + d['l'] + d['d']
        if t == 0:
            return
        wr = d['w'] / t * 100
        al = np.mean([l['total'] for l in self.stats['loss_history'][-20:]]) if self.stats['loss_history'] else 0
        vl = np.mean([l['value'] for l in self.stats['loss_history'][-20:]]) if self.stats['loss_history'] else 0
        pl = np.mean([l['policy'] for l in self.stats['loss_history'][-20:]]) if self.stats['loss_history'] else 0
        al_game = np.mean(d['lengths'][-20:]) if d['lengths'] else 0
        print(f'D{depth} | W:{d["w"]:2d} ({wr:5.1f}%) L:{d["l"]:2d} D:{d["d"]:2d} | Loss:{al:.6f} (V:{vl:.6f} P:{pl:.6f}) | AvgLen:{al_game:.0f}')

stats = Stats(drive_path)
print('Stats ready')

## Main Training Loop

In [ ]:
WIN_THRESHOLD = 0.55
GAMES_PER_DEPTH = 40
LR = 0.001
START_DEPTH = 1
MAX_DEPTH = 20

print('\n' + '='*90)
print('CHESS RL vs STOCKFISH - LC0 INSPIRED DEEP NETWORK')
print('='*90)
print(f'Win threshold: {WIN_THRESHOLD*100:.0f}% | Games/depth: {GAMES_PER_DEPTH} | LR: {LR} | Max depth: {MAX_DEPTH}')
print('='*90 + '\n')

checkpoints = sorted(drive_path.glob('ckpt_*.pt'))
start_ep = 0
curr_depth = START_DEPTH

if checkpoints:
    print(f'Loading: {checkpoints[-1].name}')
    net, optimizer, start_ep, curr_depth = load_checkpoint(checkpoints[-1])
    print(f'Resumed: ep {start_ep}, depth {curr_depth}')
    stats.print_depth(curr_depth)
    print()
else:
    net = ChessNetLc0(filters=128, num_blocks=10).to(device)
    optimizer = optim.Adam(net.parameters(), lr=LR)
    net.eval()
    print('Fresh start\n')

stockfish = StockfishBot(depth=curr_depth)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.7)

ep = start_ep
verbose_every = 50

try:
    while ep < start_ep + 1000:
        nn_w = (ep % 2 == 0)
        verbose = (ep % verbose_every == 0)
        
        winner, moves, game_len = play_game(net, encoder, stockfish, nn_is_white=nn_w, verbose=verbose)
        
        if moves:
            avg_nn_val = np.mean([m['nn_value'] for m in moves])
        else:
            avg_nn_val = 0.0
        
        loss, vloss, ploss = train_on_game(net, optimizer, moves, winner, LR)
        stats.record(winner, curr_depth, loss, vloss, ploss, game_len, avg_nn_val)
        
        ep += 1
        
        if ep % 10 == 0:
            result = 'W' if winner in [0, 1] else ('D' if winner == 2 else 'L')
            print(f'Ep {ep:4d} | R:{result} | L:{game_len:3d} | Loss:{loss:.6f} V:{vloss:.6f} P:{ploss:.6f} | AvgVal:{avg_nn_val:.4f} | ', end='')
            stats.print_depth(curr_depth)
        
        games_at_d = stats.stats['depths'].get(str(curr_depth), {}).get('games', 0)
        if games_at_d >= GAMES_PER_DEPTH:
            wr = stats.get_wr(curr_depth)
            if wr >= WIN_THRESHOLD and curr_depth < MAX_DEPTH:
                new_d = curr_depth + 1
                stockfish.set_depth(new_d)
                curr_depth = new_d
                print(f'\n🚀 LEVEL UP: {new_d-1} -> {new_d} (WR: {wr*100:.1f}%)\n')
        
        if ep % 50 == 0:
            save_checkpoint(net, optimizer, ep, curr_depth, drive_path)
            stats.save()
            print(f'\n💾 Saved ep {ep}\n')
        
        scheduler.step()

except KeyboardInterrupt:
    print(f'\n\nInterrupted ep {ep}')
    save_checkpoint(net, optimizer, ep, curr_depth, drive_path)
    stats.save()
    print('Saved')

finally:
    stockfish.quit()
    print('\nEngine closed')

print('\n' + '='*90)
print('TRAINING COMPLETE')
print('='*90)
print(f'Games: {stats.stats["total_games"]} | W:{stats.stats["total_w"]} L:{stats.stats["total_l"]} D:{stats.stats["total_d"]}')
print(f'Final depth: {curr_depth}')
print('='*90)

## Visualization

In [ ]:
with open(drive_path / 'stats.json') as f:
    s = json.load(f)

if s['total_games'] > 10:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    ax = axes[0, 0]
    ax.pie([s['total_w'], s['total_l'], s['total_d']], labels=['W', 'L', 'D'], autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c', '#95a5a6'])
    ax.set_title(f'Results ({s["total_games"]} games)')
    
    ax = axes[0, 1]
    losses = [l['total'] for l in s['loss_history']]
    ax.plot(losses, alpha=0.7, linewidth=1)
    ax.set_title('Total Loss')
    ax.grid(True, alpha=0.3)
    
    ax = axes[1, 0]
    depths = sorted([int(d) for d in s['depths'].keys()])
    wrs = []
    for d in depths:
        g = s['depths'][str(d)]
        t = g['w'] + g['l'] + g['d']
        wrs.append(g['w'] / t if t > 0 else 0)
    ax.plot(depths, wrs, marker='o', linewidth=2, markersize=8, color='#3498db')
    ax.axhline(y=0.55, color='g', linestyle='--', alpha=0.5)
    ax.set_xlabel('Depth')
    ax.set_ylabel('Win Rate')
    ax.set_title('Win Rate by Depth')
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])
    
    ax = axes[1, 1]
    games = [s['depths'][str(d)]['games'] for d in depths]
    ax.bar(depths, games, color='#9b59b6', alpha=0.7)
    ax.set_xlabel('Depth')
    ax.set_ylabel('Games')
    ax.set_title('Games per Depth')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(drive_path / 'results.png', dpi=150)
    plt.show()
    print('Saved results.png')